In [1]:
import os

import numpy as np
import pandas as pd
from scipy.stats import kurtosis

# ------------------------------------------------------------------------------
# 1. LOAD RAW PARQUET DATASET FROM ROOT DATA FOLDER
# ------------------------------------------------------------------------------
input_path = "../data/raw/market_data_raw.parquet"
df = pd.read_parquet(input_path)

print(f"Loaded {len(df):,} records for {df['Ticker'].nunique()} tickers.")


# ------------------------------------------------------------------------------
# 2. DEFINE ROLLING CALCULATION FUNCTIONS
# ------------------------------------------------------------------------------
def compute_rolling_drawdown(series, window=21):
    """
    Calculates rolling peak-to-trough drawdown over a window of trading days (~21 days = 30 calendar days).
    """
    rolling_max = series.rolling(window=window, min_periods=window).max()
    drawdown = (series - rolling_max) / rolling_max
    return drawdown


def compute_rolling_kurtosis(returns, window=63):
    """
    Calculates rolling excess kurtosis over a lookback window (e.g. 63 trading days ~ 3 months).
    """
    return returns.rolling(window=window, min_periods=window).apply(
        lambda x: kurtosis(x, fisher=True), raw=True
    )


# ------------------------------------------------------------------------------
# 3. FEATURE EXTRACTION LOOP BY TICKER
# ------------------------------------------------------------------------------
processed_frames = []

for ticker, group in df.groupby("Ticker"):
    group = group.sort_values("Date").copy()

    # Log returns
    returns = group["Log_Return"].dropna()

    # Feature 1: Rolling 30-Day Annualized Volatility (21 trading days)
    group["Vol_30D_Ann"] = returns.rolling(window=21).std() * np.sqrt(252)

    # Feature 2: Rolling 63-Day Excess Kurtosis (Tail Thickness)
    group["Kurtosis_63D"] = compute_rolling_kurtosis(returns, window=63)

    # Target Variable: 21-Day Rolling Max Drawdown
    group["Max_Drawdown_30D"] = compute_rolling_drawdown(group["Adj_Close"], window=21)

    # Binary Label: Institutional Distress (Drawdown >= 40% -> 1, else 0)
    group["Distress_Label"] = (group["Max_Drawdown_30D"].abs() >= 0.40).astype(int)

    processed_frames.append(group)

master_features = pd.concat(processed_frames, ignore_index=True)

# ------------------------------------------------------------------------------
# 4. SAVE PROCESSED DATASET TO DATA/PROCESSED/
# ------------------------------------------------------------------------------
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

output_parquet = os.path.join(output_dir, "market_data_features.parquet")
master_features.to_parquet(output_parquet, index=False)

print(f"\n[Saved] Processed dataset with features written to: {output_parquet}")
print(
    f"Total Distress Events Flagged (Label=1): {master_features['Distress_Label'].sum():,}"
)

Loaded 79,065 records for 19 tickers.

[Saved] Processed dataset with features written to: ../data/processed/market_data_features.parquet
Total Distress Events Flagged (Label=1): 875


In [4]:
import pandas as pd

# Load your processed feature dataset
df_features = pd.read_parquet("../data/processed/market_data_features.parquet")

# See breakdown of distress event counts by ticker
distress_summary = (
    df_features[df_features["Distress_Label"] == 1]
    .groupby(["Ticker", "Name", "Category"])
    .size()
    .reset_index(name="Distress_Days_Count")
    .sort_values(by="Distress_Days_Count", ascending=False)
)

print("Distress Events by Institution:")
print(distress_summary.to_string(index=False))

Distress Events by Institution:
Ticker                         Name         Category  Distress_Days_Count
  ^VIX        CBOE Volatility Index        Benchmark                  199
  FRCB          First Republic Bank  Distressed_2023                  161
   AIG American International Group  Distressed_2008                   96
     C                    Citigroup  Distressed_2008                   66
  FITB          Fifth Third Bancorp    Regional_Bank                   60
   WAL             Western Alliance    Regional_Bank                   57
   BAC              Bank of America Anchor_Financial                   48
   IVR     Invesco Mortgage Capital  Distressed_2020                   41
   KEY                      KeyCorp    Regional_Bank                   37
    MS               Morgan Stanley Anchor_Financial                   29
   MFA                MFA Financial  Distressed_2020                   28
   TWO       Two Harbors Investment  Distressed_2020                   23
    GS

In [ ]:
import pandas as pd

# Load raw data
df_raw = pd.read_parquet("../data/raw/market_data_raw.parquet")

# Check date ranges and record counts for Lehman and SVB
check_tickers = ["LEHMQ", "SIVB", "BSC"]
subset = df_raw[df_raw["Ticker"].isin(check_tickers)]

print(subset.groupby("Ticker")["Date"].agg(["min", "max", "count"]))

Empty DataFrame
Columns: [min, max, count]
Index: []
